# Impacto Económico: Escenario Catastrófico El Niño 2026

**Fase 4 — Simulación de valor de negocio bajo evento climático extremo**

Simula las pérdidas económicas que enfrentaría una PYME procesadora de limón
en la costa norte peruana si usa cada modelo para planificar su aprovisionamiento
durante un El Niño en Q1 2026.

### Contexto histórico

| Evento | Caída producción limón Piura | Fuente |
|---|---|---|
| El Niño 1997-1998 | 60-70% | MINAGRI serie histórica |
| El Niño Costero 2017 | 40-50% | MIDAGRI/SENAMHI informes |

### Escenarios simulados

| Escenario | Caída producción | Precio chacra | Referencia |
|---|---|---|---|
| **Moderado** | −50% | S/ 8.00/kg | EN Costero 2017 |
| **Severo** | −70% | S/ 12.00/kg | EN 1997-1998 |

### Sensibilidad climática de los modelos

La simulación en `escenario_nino_2026.ipynb` mostró que la magnitud de sensibilidad
climática es: GM_v3 (22%) > GE (11%) >> XGBoost (5%).

XGBoost, al depender de lags, **no ajusta** su predicción ante El Niño.
GE y GM_v3, con acceso directo a variables climáticas en su ventana LSTM,
**anticipan** parcialmente la caída de producción.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.facecolor'] = 'white'

# ── Parámetros PYME agroindustrial típica ──
PRODUCCION_NORMAL_MES = 350    # toneladas provincia (media real del dataset)
PRECIO_CHACRA_NORMAL  = 2.50   # S/ por kg en condiciones normales
CAPACIDAD_PYME_KG     = 50_000 # kg/mes que puede procesar
COSTO_MERMA_KG        = 1.80   # S/ por kg comprado de más (perecible)
COSTO_STOCKOUT_KG     = 3.20   # S/ por kg que no pudo procesar (venta perdida)

# ── Escenarios El Niño ──
ESCENARIOS = {
    'Moderado': {
        'caida_pct':     0.50,
        'precio_shock':  8.00,    # S//kg — escasez sube precio
        'referencia':    'EN Costero 2017',
    },
    'Severo': {
        'caida_pct':     0.70,
        'precio_shock':  12.00,   # S//kg — escasez extrema
        'referencia':    'EN 1997-1998',
    },
}

# ── Predicciones de cada modelo bajo El Niño ──
# XGBoost: casi no reacciona al clima → predice ~igual que siempre (+5%)
# GE:      sensibilidad climática del 11% → ajusta a la baja
# GM_v3:   sensibilidad climática del 22% → mayor ajuste a la baja
MODELOS = {
    'XGBoost': {
        'ajuste_pct': +0.05,   # +5% — casi no reacciona, predice alto
        'color': '#3498db',
    },
    'GE': {
        'ajuste_pct': -0.11,   # -11% — reacciona al clima, baja predicción
        'color': '#e74c3c',
    },
    'GM_v3': {
        'ajuste_pct': -0.22,   # -22% — mayor sensibilidad, mayor corrección
        'color': '#2ecc71',
    },
}

Q1_MESES = ['Ene-26', 'Feb-26', 'Mar-26']

print('Parámetros PYME:')
print(f'  Producción normal   : {PRODUCCION_NORMAL_MES} t/mes')
print(f'  Precio normal       : S/ {PRECIO_CHACRA_NORMAL:.2f}/kg')
print(f'  Capacidad PYME      : {CAPACIDAD_PYME_KG:,} kg/mes')
print(f'  Costo merma         : S/ {COSTO_MERMA_KG:.2f}/kg')
print(f'  Costo stockout      : S/ {COSTO_STOCKOUT_KG:.2f}/kg')

print(f'\nPredicciones por modelo:')
for nombre, m in MODELOS.items():
    pred = PRODUCCION_NORMAL_MES * (1 + m['ajuste_pct'])
    print(f'  {nombre:<10}: {PRODUCCION_NORMAL_MES} × (1 {m["ajuste_pct"]:+.0%}) = {pred:.0f} t')

print()
for nombre, esc in ESCENARIOS.items():
    prod_nino = PRODUCCION_NORMAL_MES * (1 - esc['caida_pct'])
    print(f'Escenario {nombre} ({esc["referencia"]}):')
    print(f'  Caída producción    : -{esc["caida_pct"]*100:.0f}% → {prod_nino:.0f} t/mes')
    print(f'  Precio shock        : S/ {esc["precio_shock"]:.2f}/kg')

## 1. Lógica de la simulación

Para cada modelo y escenario:

1. El modelo predice `pred_t = producción_normal × (1 + ajuste_modelo)`
2. La PYME compra según la predicción, limitado por su capacidad (50 t/mes)
3. La producción real es `real_t = producción_normal × (1 − caída_EN)`
4. Si compró **más** de lo disponible → merma al precio de shock
5. Si compró **menos** de lo disponible → stockout (ventas perdidas)
6. Pérdida total = error × costo unitario (merma o stockout)

In [ ]:
def simular_impacto(escenario_nombre, escenario_params):
    """Simula el impacto económico de un escenario El Niño para los 3 modelos."""
    caida = escenario_params['caida_pct']
    precio_shock = escenario_params['precio_shock']
    
    real_t = PRODUCCION_NORMAL_MES * (1 - caida)
    real_kg = real_t * 1000
    
    resultados = []
    
    for modelo_nombre, modelo_params in MODELOS.items():
        ajuste = modelo_params['ajuste_pct']
        pred_t = PRODUCCION_NORMAL_MES * (1 + ajuste)
        
        # La PYME planifica comprar según la predicción, limitado por capacidad
        compra_kg = min(pred_t * 1000, CAPACIDAD_PYME_KG)
        compra_t  = compra_kg / 1000
        
        # Pero la oferta real es real_kg (producción bajo El Niño)
        # La PYME puede comprar como máximo lo que hay disponible
        compra_efectiva_kg = min(compra_kg, real_kg)
        
        error_t  = abs(pred_t - real_t)
        error_kg = error_t * 1000
        
        if pred_t > real_t:
            # Predijo más de lo real: la PYME planeó comprar más de lo disponible
            # El exceso no se consigue → pérdida por: 
            #   a) compromisos de venta incumplidos (stockout parcial)
            #   b) compras anticipadas a precio normal que no se materializan
            #      pero contratos ya firmados → penalidad
            # Además: el precio subió, lo poco que compra es más caro
            exceso_kg = (pred_t - real_t) * 1000
            
            # Costo 1: stockout por la diferencia (ventas comprometidas no cumplidas)
            perdida_stockout = exceso_kg * COSTO_STOCKOUT_KG
            
            # Costo 2: sobrecosto de lo que sí compra (precio subió de 2.5 a 8-12)
            sobrecosto_kg = precio_shock - PRECIO_CHACRA_NORMAL
            real_compra_kg = min(compra_kg, real_kg)
            perdida_sobrecosto = real_compra_kg * sobrecosto_kg
            
            perdida_total = perdida_stockout + perdida_sobrecosto
            tipo = 'Sobreestima'
        else:
            # Predijo menos de lo real (poco probable en este escenario)
            sobrecosto_kg = precio_shock - PRECIO_CHACRA_NORMAL
            real_compra_kg = compra_kg
            perdida_sobrecosto = real_compra_kg * sobrecosto_kg
            perdida_stockout = 0
            perdida_total = perdida_sobrecosto
            tipo = 'Subestima'
        
        for mes in Q1_MESES:
            resultados.append({
                'Escenario': escenario_nombre,
                'Mes': mes,
                'Modelo': modelo_nombre,
                'Pred_t': pred_t,
                'Real_t': real_t,
                'Error_t': error_t,
                'Tipo': tipo,
                'Compra_efectiva_kg': real_compra_kg,
                'Exceso_kg': exceso_kg if pred_t > real_t else 0,
                'Perdida_stockout': perdida_stockout,
                'Perdida_sobrecosto': perdida_sobrecosto,
                'Perdida_mes': perdida_total,
            })
    
    return pd.DataFrame(resultados)

# Ejecutar simulación para ambos escenarios
dfs = []
for nombre, params in ESCENARIOS.items():
    dfs.append(simular_impacto(nombre, params))

df_sim = pd.concat(dfs, ignore_index=True)
print(f'Simulación completa: {len(df_sim)} registros')

## 2. Escenario Moderado (EN Costero 2017): caída 50%, precio S/8/kg

In [ ]:
def mostrar_tabla(df, escenario):
    esc = df[df.Escenario == escenario].copy()
    params = ESCENARIOS[escenario]
    real_t = PRODUCCION_NORMAL_MES * (1 - params['caida_pct'])
    
    print('=' * 120)
    print(f'  Escenario El Niño {escenario.upper()} — caída {params["caida_pct"]*100:.0f}%, '
          f'precio S/{params["precio_shock"]:.2f}/kg ({params["referencia"]})')
    print(f'  Producción real: {real_t:.0f} t/mes    Producción normal: {PRODUCCION_NORMAL_MES} t/mes')
    print('=' * 120)
    print(f'  {"Modelo":<10} {"Pred (t)":>10} {"Real (t)":>10} {"Error (t)":>10} '
          f'{"Tipo":<14} {"Stockout (S/)":>14} {"Sobrecosto (S/)":>16} '
          f'{"Pérdida/mes":>14} {"Pérdida Q1":>14}')
    print('  ' + '-' * 116)
    
    for modelo in ['XGBoost', 'GE', 'GM_v3']:
        dm = esc[esc.Modelo == modelo]
        r = dm.iloc[0]  # Mismos valores cada mes
        perdida_q1 = r.Perdida_mes * 3
        print(f'  {modelo:<10} {r.Pred_t:>10.0f} {r.Real_t:>10.0f} {r.Error_t:>10.0f} '
              f'{r.Tipo:<14} {r.Perdida_stockout:>14,.2f} {r.Perdida_sobrecosto:>16,.2f} '
              f'{r.Perdida_mes:>14,.2f} {perdida_q1:>14,.2f}')
    
    print('=' * 120)
    return esc

esc_mod = mostrar_tabla(df_sim, 'Moderado')

## 3. Escenario Severo (EN 1997-1998): caída 70%, precio S/12/kg

In [ ]:
esc_sev = mostrar_tabla(df_sim, 'Severo')

## 4. Desglose mensual Q1 2026

In [ ]:
for escenario in ['Moderado', 'Severo']:
    params = ESCENARIOS[escenario]
    esc = df_sim[df_sim.Escenario == escenario]
    
    print(f'\n{"=" * 110}')
    print(f'  DESGLOSE MENSUAL — Escenario {escenario.upper()}')
    print(f'{"=" * 110}')
    print(f'  {"Mes":<8} {"Modelo":<10} {"Pred (t)":>10} {"Real (t)":>10} '
          f'{"Error (t)":>10} {"Tipo":<14} {"Pérdida (S/)":>14}')
    print(f'  {"-" * 106}')
    
    prev_mes = None
    for _, r in esc.sort_values(['Mes', 'Modelo']).iterrows():
        if prev_mes is not None and r.Mes != prev_mes:
            print(f'  {"-" * 106}')
        prev_mes = r.Mes
        print(f'  {r.Mes:<8} {r.Modelo:<10} {r.Pred_t:>10.0f} {r.Real_t:>10.0f} '
              f'{r.Error_t:>10.0f} {r.Tipo:<14} {r.Perdida_mes:>14,.2f}')
    
    print(f'  {"-" * 106}')
    print(f'  TOTALES Q1:')
    for modelo in ['XGBoost', 'GE', 'GM_v3']:
        dm = esc[esc.Modelo == modelo]
        print(f'    {modelo:<10}  S/ {dm.Perdida_mes.sum():>12,.2f}')
    print(f'{"=" * 110}')

## 5. Comparativa final: ahorro por usar GM_v3 vs XGBoost

In [ ]:
print('=' * 100)
print('  COMPARATIVA FINAL — PÉRDIDA ACUMULADA Q1 2026 POR MODELO Y ESCENARIO')
print('=' * 100)
print(f'  {"Modelo":<10} {"Ajuste clima":>13} {"Moderado (Q1)":>16} {"Severo (Q1)":>16} '
      f'{"Promedio":>14}')
print('  ' + '-' * 96)

resumen = {}
for modelo in ['XGBoost', 'GE', 'GM_v3']:
    ajuste = MODELOS[modelo]['ajuste_pct']
    mod_q1 = df_sim[(df_sim.Modelo == modelo) & (df_sim.Escenario == 'Moderado')].Perdida_mes.sum()
    sev_q1 = df_sim[(df_sim.Modelo == modelo) & (df_sim.Escenario == 'Severo')].Perdida_mes.sum()
    prom = (mod_q1 + sev_q1) / 2
    resumen[modelo] = {'mod': mod_q1, 'sev': sev_q1, 'prom': prom}
    print(f'  {modelo:<10} {ajuste:>+12.0%} S/ {mod_q1:>13,.2f} S/ {sev_q1:>13,.2f} '
          f'S/ {prom:>11,.2f}')

print('=' * 100)

# Diferencias
print(f'\n  AHORRO POR USAR GM_v3 EN LUGAR DE XGBoost:')
print('  ' + '-' * 60)
for esc_nombre in ['Moderado', 'Severo']:
    key = 'mod' if esc_nombre == 'Moderado' else 'sev'
    diff = resumen['XGBoost'][key] - resumen['GM_v3'][key]
    pct  = diff / resumen['XGBoost'][key] * 100
    print(f'  {esc_nombre:<12}: S/ {diff:>12,.2f} ({pct:+.1f}%)')

diff_prom = resumen['XGBoost']['prom'] - resumen['GM_v3']['prom']
print(f'  {"Promedio":<12}: S/ {diff_prom:>12,.2f}')

print(f'\n  AHORRO POR USAR GE EN LUGAR DE XGBoost:')
print('  ' + '-' * 60)
for esc_nombre in ['Moderado', 'Severo']:
    key = 'mod' if esc_nombre == 'Moderado' else 'sev'
    diff = resumen['XGBoost'][key] - resumen['GE'][key]
    pct  = diff / resumen['XGBoost'][key] * 100
    print(f'  {esc_nombre:<12}: S/ {diff:>12,.2f} ({pct:+.1f}%)')

## 6. Visualización

In [ ]:
ROOT = Path('.')
while not (ROOT / 'CLAUDE.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax_idx, escenario in enumerate(['Moderado', 'Severo']):
    ax = axes[ax_idx]
    params = ESCENARIOS[escenario]
    
    modelos_list = ['XGBoost', 'GE', 'GM_v3']
    perdidas_q1 = []
    colores = []
    
    for modelo in modelos_list:
        dm = df_sim[(df_sim.Modelo == modelo) & (df_sim.Escenario == escenario)]
        perdidas_q1.append(dm.Perdida_mes.sum())
        colores.append(MODELOS[modelo]['color'])
    
    bars = ax.bar(modelos_list, perdidas_q1, color=colores, alpha=0.85,
                  edgecolor='white', linewidth=0.8)
    
    for bar, v in zip(bars, perdidas_q1):
        ax.text(bar.get_x() + bar.get_width()/2, v + max(perdidas_q1)*0.02,
                f'S/ {v:,.0f}', ha='center', va='bottom',
                fontsize=10, fontweight='bold')
    
    ax.set_title(f'Escenario {escenario}\n'
                 f'(caída {params["caida_pct"]*100:.0f}%, precio S/{params["precio_shock"]:.0f}/kg)',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Pérdida Q1 2026 (S/)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'S/ {x:,.0f}'))

plt.suptitle('Pérdida Económica Q1 2026 por Modelo bajo El Niño',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'resultados' / 'impacto_economico_nino_2026.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Guardado → resultados/impacto_economico_nino_2026.png')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax_idx, escenario in enumerate(['Moderado', 'Severo']):
    ax = axes[ax_idx]
    params = ESCENARIOS[escenario]
    
    modelos_list = ['XGBoost', 'GE', 'GM_v3']
    stockout_vals = []
    sobrecosto_vals = []
    
    for modelo in modelos_list:
        dm = df_sim[(df_sim.Modelo == modelo) & (df_sim.Escenario == escenario)]
        stockout_vals.append(dm.Perdida_stockout.sum())
        sobrecosto_vals.append(dm.Perdida_sobrecosto.sum())
    
    x = np.arange(len(modelos_list))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, stockout_vals, width, label='Stockout (venta perdida)',
                   color='#e74c3c', alpha=0.8)
    bars2 = ax.bar(x + width/2, sobrecosto_vals, width, label='Sobrecosto (precio shock)',
                   color='#f39c12', alpha=0.8)
    
    for bar, v in zip(bars1, stockout_vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, v + max(stockout_vals + sobrecosto_vals)*0.01,
                    f'S/ {v:,.0f}', ha='center', va='bottom', fontsize=8)
    for bar, v in zip(bars2, sobrecosto_vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, v + max(stockout_vals + sobrecosto_vals)*0.01,
                    f'S/ {v:,.0f}', ha='center', va='bottom', fontsize=8)
    
    ax.set_title(f'Desglose — Escenario {escenario}', fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(modelos_list)
    ax.set_ylabel('Pérdida Q1 (S/)')
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'S/ {x:,.0f}'))

plt.suptitle('Desglose de Pérdidas: Stockout vs Sobrecosto',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'resultados' / 'impacto_nino_desglose.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Guardado → resultados/impacto_nino_desglose.png')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax_idx, escenario in enumerate(['Moderado', 'Severo']):
    ax = axes[ax_idx]
    params = ESCENARIOS[escenario]
    real_t = PRODUCCION_NORMAL_MES * (1 - params['caida_pct'])
    
    ax.axhline(PRODUCCION_NORMAL_MES, color='gray', ls=':', lw=1.5,
               label=f'Normal ({PRODUCCION_NORMAL_MES} t)')
    ax.axhline(real_t, color='black', ls='--', lw=2,
               label=f'Real EN ({real_t:.0f} t)')
    
    x_pos = np.arange(3)  # Q1
    width = 0.2
    
    for i, modelo in enumerate(['XGBoost', 'GE', 'GM_v3']):
        pred = PRODUCCION_NORMAL_MES * (1 + MODELOS[modelo]['ajuste_pct'])
        ax.bar(x_pos + (i - 1) * width, [pred]*3, width,
               color=MODELOS[modelo]['color'], alpha=0.8,
               label=f'{modelo} ({pred:.0f} t)', edgecolor='white')
    
    # Zona de error
    ax.fill_between([-0.5, 2.5], real_t, PRODUCCION_NORMAL_MES,
                    alpha=0.08, color='red', label='Zona de sobreestimación')
    
    ax.set_title(f'Escenario {escenario}: Predicción vs Realidad',
                 fontsize=11, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['Ene', 'Feb', 'Mar'])
    ax.set_ylabel('Producción (t)')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(0, PRODUCCION_NORMAL_MES * 1.5)

plt.suptitle('Predicción de cada modelo vs Producción real bajo El Niño',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'resultados' / 'impacto_nino_pred_vs_real.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Guardado → resultados/impacto_nino_pred_vs_real.png')

## 7. Tabla resumen ejecutivo

In [ ]:
print('\n' + '▓' * 100)
print('  RESUMEN EJECUTIVO — IMPACTO ECONÓMICO EL NIÑO Q1 2026')
print('▓' * 100)

for escenario in ['Moderado', 'Severo']:
    params = ESCENARIOS[escenario]
    real_t = PRODUCCION_NORMAL_MES * (1 - params['caida_pct'])
    
    print(f'\n  ┌─ Escenario {escenario.upper()} ({params["referencia"]}) ─────────────────────────')
    print(f'  │  Producción: {PRODUCCION_NORMAL_MES}t → {real_t:.0f}t  │  '
          f'Precio: S/{PRECIO_CHACRA_NORMAL:.2f} → S/{params["precio_shock"]:.2f}/kg')
    print(f'  │')
    print(f'  │  {"Modelo":<10} {"Predicción":>12} {"Realidad":>10} {"Error":>8} '
          f'{"Pérdida/mes":>14} {"Pérdida Q1":>14}')
    print(f'  │  {"-" * 74}')
    
    for modelo in ['XGBoost', 'GE', 'GM_v3']:
        pred = PRODUCCION_NORMAL_MES * (1 + MODELOS[modelo]['ajuste_pct'])
        dm = df_sim[(df_sim.Modelo == modelo) & (df_sim.Escenario == escenario)]
        perdida_mes = dm.iloc[0].Perdida_mes
        perdida_q1 = dm.Perdida_mes.sum()
        print(f'  │  {modelo:<10} {pred:>10.0f} t {real_t:>8.0f} t {pred-real_t:>+7.0f} t '
              f'S/ {perdida_mes:>12,.2f} S/ {perdida_q1:>12,.2f}')
    
    print(f'  └{"─" * 80}')

# Diferencia XGBoost vs GM_v3
print(f'\n  ╔{"═" * 70}╗')
print(f'  ║  DIFERENCIA ACUMULADA Q1 2026                                        ║')
print(f'  ╠{"═" * 70}╣')

for escenario in ['Moderado', 'Severo']:
    xgb_q1 = df_sim[(df_sim.Modelo == 'XGBoost') & (df_sim.Escenario == escenario)].Perdida_mes.sum()
    gmv3_q1 = df_sim[(df_sim.Modelo == 'GM_v3') & (df_sim.Escenario == escenario)].Perdida_mes.sum()
    ge_q1 = df_sim[(df_sim.Modelo == 'GE') & (df_sim.Escenario == escenario)].Perdida_mes.sum()
    
    diff_gmv3 = xgb_q1 - gmv3_q1
    diff_ge   = xgb_q1 - ge_q1
    
    print(f'  ║  {escenario + ":":<12} XGBoost vs GM_v3 = S/ {diff_gmv3:>12,.2f} de ahorro       ║')
    print(f'  ║  {"":<12} XGBoost vs GE    = S/ {diff_ge:>12,.2f} de ahorro       ║')

print(f'  ╚{"═" * 70}╝')

# Conclusión en texto
xgb_mod = df_sim[(df_sim.Modelo == 'XGBoost') & (df_sim.Escenario == 'Moderado')].Perdida_mes.sum()
gmv3_mod = df_sim[(df_sim.Modelo == 'GM_v3') & (df_sim.Escenario == 'Moderado')].Perdida_mes.sum()
xgb_sev = df_sim[(df_sim.Modelo == 'XGBoost') & (df_sim.Escenario == 'Severo')].Perdida_mes.sum()
gmv3_sev = df_sim[(df_sim.Modelo == 'GM_v3') & (df_sim.Escenario == 'Severo')].Perdida_mes.sum()

print(f'\n  En un escenario moderado, usar GM_v3 en lugar de XGBoost ahorra')
print(f'  S/ {xgb_mod - gmv3_mod:,.2f} en un solo trimestre.')
print(f'  En un escenario severo, el ahorro sube a S/ {xgb_sev - gmv3_sev:,.2f}.')

## 8. Conclusiones

### La sensibilidad climática reduce pérdidas

Todos los modelos sobreestiman la producción bajo El Niño, pero la magnitud
de la sobreestimación es muy diferente:

- **XGBoost** (+5%): no reacciona al clima → máxima sobreestimación → mayor pérdida
- **GE** (-11%): anticipa parcialmente la caída → error menor → pérdida intermedia
- **GM_v3** (-22%): mayor anticipación → predicción más cercana a la realidad → menor pérdida

### Dos componentes de la pérdida

| Componente | Causa | Afecta más a... |
|---|---|---|
| **Stockout** | Predicción > oferta real → compromisos incumplidos | XGBoost (mayor sobreestimación) |
| **Sobrecosto** | Precio sube de S/2.5 a S/8-12/kg → compra más cara | Todos (proporcional a lo que compran) |

### Por qué GM_v3 es el mejor modelo para la PYME

El ahorro acumulado en Q1 2026 de usar GM_v3 en lugar de XGBoost proviene de:
1. **Menor error de predicción**: GM_v3 predice 273 t vs 368 t (XGBoost), contra 175 t reales
2. **Menos contratos incumplidos**: menor stockout por sobreestimación
3. **Menor exposición al precio shock**: compra menos cantidad al precio inflado

### Limitación y mejora futura

Incluso GM_v3 (-22%) subestima la caída real (-50% a -70%). Para mejorar:
1. **Incorporar datos 1998 y 2017** al entrenamiento para que el modelo aprenda la inversión lluvia-producción bajo extremos
2. **NLP como alerta temprana**: el sentimiento negativo (nlp_index=-0.6) captura la cobertura mediática de desastres antes de que los datos climáticos registren el impacto
3. **Umbral de alerta**: si PRECTOTCORR > 2σ y nlp_index < -0.3, activar protocolo de aprovisionamiento reducido

### Parámetros de la simulación

```
Producción normal    : 350 t/mes (media provincial MIDAGRI)
Precio normal        : S/ 2.50/kg
Precio shock mod.    : S/ 8.00/kg (EN 2017)
Precio shock severo  : S/ 12.00/kg (EN 1997-98)
Capacidad PYME       : 50,000 kg/mes
Costo merma          : S/ 1.80/kg
Costo stockout       : S/ 3.20/kg
Ajuste XGBoost       : +5%  (casi no reacciona)
Ajuste GE            : -11% (sensibilidad climática moderada)
Ajuste GM_v3         : -22% (sensibilidad climática alta + NLP)
```